In [4]:
%pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 13.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [jiwer]
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 9.9 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.6 MB/s  0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=bdc72497fd5780f7e8b8d635b1991b59a24f1ccd85bfd59bd595f5ccd9553a78
  Stored in directory: /Users/melaniequ/Library/Caches/pip/wheels/be/22/86/dcf050d50e02b3bb67b5a3b289be2f3376ce7f8717d012d6a8
Successfully built openai-whisper
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openai-whisper]
Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path
# Path provides an easier way to work with file and folder paths.
#
# NOTE:
# In the current code, Path is imported but the audio_file is
# still written as a string. We could use Path later if desired.


import whisper
# Imports OpenAI's Whisper speech-to-text model.
#
# Whisper takes an audio file and attempts to convert
# the spoken audio into text.


from jiwer import wer
# Imports the Word Error Rate (WER) function.
#
# WER measures how different the predicted transcription is
# from the expected/reference transcription.


# ============================================================
# LOAD WHISPER
# ============================================================

print(
    "Loading Whisper..."
)
# Lets us know that the Whisper model is being loaded.


model = whisper.load_model(
    "base"
)
# Loads the pretrained Whisper "base" model.
#
# Whisper has multiple model sizes.
# "base" is a relatively small model that is useful for
# testing transcription without requiring as much
# computing power as larger models.
#
# The model will be used to convert generated speech
# into text.


print(
    "Whisper loaded."
)
# Confirms that Whisper has finished loading.


# ============================================================
# TRANSCRIBE
# ============================================================

def transcribe(
    audio_file
):
    # Defines a function that takes an audio file
    # and converts the speech inside it into text.


    result = model.transcribe(
        str(audio_file)
    )
    # Sends the audio file to Whisper.
    #
    # Whisper performs automatic speech recognition (ASR).
    #
    # The result is a dictionary containing information
    # such as the detected language and transcription.
    #
    # str(audio_file) converts the file path to a string
    # in case a Path object is provided.


    return result["text"].strip()
    # Extracts the actual transcription from the result.
    #
    # .strip() removes unnecessary spaces or newline
    # characters from the beginning and end of the text.
    #
    # Example:
    #
    # " The quick brown fox jumps... "
    #
    # becomes:
    #
    # "The quick brown fox jumps..."


# ============================================================
# CALCULATE WER
# ============================================================

def calculate_wer(
    reference_text,
    generated_audio
):
    # Calculates how accurately the generated audio
    # represents the expected text.
    #
    # reference_text:
    # The text that SHOULD have been spoken.
    #
    # generated_audio:
    # The synthetic audio generated by XTTS-v2.


    predicted_text = transcribe(
        generated_audio
    )
    # Sends the generated audio to Whisper.
    #
    # Whisper listens to the generated speech and produces
    # its own transcription.
    #
    # This gives us the text that the generated audio
    # actually sounds like to the speech-recognition model.


    score = wer(
        reference_text,
        predicted_text
    )
    # Calculates Word Error Rate.
    #
    # reference_text = correct/expected text
    #
    # predicted_text = Whisper's transcription
    #
    # WER is based on:
    #
    # substitutions + deletions + insertions
    # ----------------------------------------
    #           number of reference words
    #
    # Interpretation:
    #
    # WER = 0.0
    # Perfect word-level match.
    #
    # Lower WER = better transcription accuracy.
    #
    # Higher WER = more differences between the expected
    # text and Whisper's transcription.


Loading Whisper...
Whisper loaded.


In [2]:
from pathlib import Path
import pandas as pd

manifest_path = Path("../data/evaluation/vc_test_manifest.csv")
manifest = pd.read_csv(manifest_path)

# Keep only rows with a usable reference transcript
manifest["reference_text"] = (
    manifest["reference_text"].fillna("").astype(str).str.strip()
)

valid_manifest = manifest[manifest["reference_text"] != ""].copy()

print("Total audio files:", len(manifest))
print("Files with reference transcripts:", len(valid_manifest))
print("Files skipped due to missing transcripts:",
      len(manifest) - len(valid_manifest))

results = []

for i, row in valid_manifest.iterrows():
    audio_path = Path(row["generated_audio"])

    if not audio_path.exists():
        print(f"Audio file not found: {audio_path}")
        continue

    reference = row["reference_text"]
    hypothesis = transcribe(audio_path)
    sample_wer = wer(reference, hypothesis)

    results.append({
        "generated_audio": str(audio_path),
        "target_speaker": row["target_speaker"],
        "reference_text": reference,
        "transcription": hypothesis,
        "wer": sample_wer
    })

    if len(results) % 25 == 0:
        print(f"Processed {len(results)} files...")

results_df = pd.DataFrame(results)

output_path = Path("../data/evaluation/vc_wer_results.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(output_path, index=False)

print("\nEvaluation complete.")
print("Files evaluated:", len(results_df))
print("Results saved to:", output_path)

display(results_df.head(10))

Total audio files: 740
Files with reference transcripts: 740
Files skipped due to missing transcripts: 0


/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  w

Processed 25 files...


/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  w

Processed 50 files...


...

/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  w

Processed 725 files...


/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  w


Evaluation complete.
Files evaluated: 740
Results saved to: ../data/evaluation/vc_wer_results.csv


,generated_audio,target_speaker,reference_text,transcription,wer
0,/Users/melaniequ/Voice Cloning and Fake Audio ...,FBCH0,Don't ask me to carry an oily rag like that.,Don't ask me to carry an oily rag like that.,0.00
1,/Users/melaniequ/Voice Cloning and Fake Audio ...,FBCH0,"Your voice is delightful, he approved with a w...","Your voice is delightful, you approved with a ...",0.10
2,/Users/melaniequ/Voice Cloning and Fake Audio ...,FBCH0,Turbulent tides rose as much as fifty feet.,turbulent tides rose as much as 50 feet.,0.25
3,/Users/melaniequ/Voice Cloning and Fake Audio ...,FBCH0,Sometimes soldiers wrote letters while bullets...,Sometimes soldiers wrote letters while bullets...,0.00
4,/Users/melaniequ/Voice Cloning and Fake Audio ...,FBCH0,Cyclical programs will never compile.,Sicklical programs will never compile.,0.20
5,/Users/melaniequ/Voice Cloning and Fake Audio ...,FBCH0,Clear pronunciation is appreciated.,Clear pronunciation is appreciated.,0.00
6,/Users/melaniequ/Voice Cloning and Fake Audio ...,FBCH0,Steve collects rare and novel coins.,Steve collects rare and novel coins.,0.00
7,/Users/melaniequ/Voice Cloning and Fake Audio ...,FBCH0,A huge power outage rarely occurs.,A huge power outage rarely occurs.,0.00
8,/Users/melaniequ/Voice Cloning and Fake Audio ...,FBCH0,Academic aptitude guarantees your diploma.,"Academic aptitude, Guarantees, or Diploma.",0.80
9,/Users/melaniequ/Voice Cloning and Fake Audio ...,FCEG0,Don't ask me to carry an oily rag like that.,Don't ask me to carry an oily rag like that.,0.00
